# 🔀 Day 04: Linked List Merge & Sort — Combining and Ordering

---

## Why Merge Sort Loves Linked Lists

Here's something interesting: merge sort on arrays needs O(n) extra space for the merge step. But on linked lists? **O(1) extra space.** You just rearrange pointers — no copying.

That makes merge sort the natural choice for sorting linked lists. Quick sort is great for arrays (cache-friendly, in-place), but for linked lists, merge sort is king.

The building blocks you need:
1. **Find the middle** (fast/slow pointer — Day 02)
2. **Merge two sorted lists** (Day 01)
3. **Recursion** (split → sort each half → merge)

You already know 1 and 2. This module ties it all together.

---

In [ ]:
# Helpers
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

def print_list(head):
    parts = []
    while head:
        parts.append(str(head.val))
        head = head.next
    print(" → ".join(parts) + " → None")

---

## 🎯 Problem 1: Sort List (LC #148)

Sort a linked list in O(n log n) time and O(1) space (constant extra space).

```
Input:  4 → 2 → 1 → 3
Output: 1 → 2 → 3 → 4
```

**Merge sort on linked list — the full recipe:**

```
1. Find middle (fast/slow)
2. Split list into two halves (cut the link at middle)
3. Recursively sort each half
4. Merge the two sorted halves
```

This is one of those problems where everything from the previous modules clicks together.

In [ ]:
def sortList(head):
    """
    Merge sort on linked list.
    Time: O(n log n), Space: O(log n) recursion stack
    """
    # Base case: 0 or 1 node is already sorted
    if not head or not head.next:
        return head
    
    # Step 1: Find middle (use slow/fast, but stop slow BEFORE middle)
    slow, fast = head, head.next  # Note: fast starts at head.next!
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    
    # Step 2: Split — cut the link
    mid = slow.next
    slow.next = None  # First half ends here
    
    # Step 3: Recursively sort both halves
    left = sortList(head)
    right = sortList(mid)
    
    # Step 4: Merge
    return merge(left, right)

def merge(l1, l2):
    """Merge two sorted lists — our trusty merge from Day 01."""
    dummy = ListNode(0)
    tail = dummy
    while l1 and l2:
        if l1.val <= l2.val:
            tail.next = l1
            l1 = l1.next
        else:
            tail.next = l2
            l2 = l2.next
        tail = tail.next
    tail.next = l1 or l2
    return dummy.next

lst = build_list([4, 2, 1, 3])
print("Before:", end=" "); print_list(build_list([4,2,1,3]))
sorted_lst = sortList(lst)
print("After: ", end=" "); print_list(sorted_lst)  # 1→2→3→4

lst2 = build_list([-1, 5, 3, 4, 0])
print("Sorted:", end=" "); print_list(sortList(lst2))  # -1→0→3→4→5

**Important subtlety:** When finding the middle for splitting, we start `fast = head.next` (not `head`). This ensures for a 2-node list like `[1,2]`, `slow` stays at `1` so we split into `[1]` and `[2]` instead of having an infinite recursion.

---

## 🎯 Problem 2: Merge K Sorted Lists (LC #23) — Hard

You have k sorted linked lists. Merge them all into one sorted list.

```
Input: [[1→4→5], [1→3→4], [2→6]]
Output: 1→1→2→3→4→4→5→6
```

**Three approaches:**

| Approach | Time | Space |
|----------|------|-------|
| Merge one by one | O(kN) | O(1) |
| **Min-heap** | **O(N log k)** | **O(k)** |
| **Divide & conquer** | **O(N log k)** | **O(log k)** |

The heap approach: push the head of each list into a min-heap. Pop the smallest, push its next node. Repeat.

In [ ]:
import heapq

def mergeKLists(lists):
    """
    Merge k sorted lists using a min-heap.
    Time: O(N log k) where N = total nodes, k = number of lists
    Space: O(k) for the heap
    """
    dummy = ListNode(0)
    tail = dummy
    
    # Push first node of each list into heap
    # (value, index_tiebreaker, node) — index breaks ties for heapq
    heap = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst.val, i, lst))
    
    idx = len(lists)  # Tiebreaker counter
    
    while heap:
        val, _, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        
        if node.next:
            heapq.heappush(heap, (node.next.val, idx, node.next))
            idx += 1
    
    return dummy.next

l1 = build_list([1, 4, 5])
l2 = build_list([1, 3, 4])
l3 = build_list([2, 6])
result = mergeKLists([l1, l2, l3])
print("Merged K:", end=" "); print_list(result)  # 1→1→2→3→4→4→5→6

In [ ]:
# Alternative: Divide & Conquer approach
# Pair up lists and merge pairs, like a tournament bracket

def mergeKLists_dc(lists):
    """
    Divide & Conquer: pair up and merge.
    Time: O(N log k), Space: O(log k)
    """
    if not lists:
        return None
    
    while len(lists) > 1:
        merged = []
        for i in range(0, len(lists), 2):
            l1 = lists[i]
            l2 = lists[i + 1] if i + 1 < len(lists) else None
            merged.append(merge(l1, l2))
        lists = merged
    
    return lists[0]

l1 = build_list([1, 4, 5])
l2 = build_list([1, 3, 4])
l3 = build_list([2, 6])
result = mergeKLists_dc([l1, l2, l3])
print("D&C merge:", end=" "); print_list(result)

---

## 🎯 Problem 3: Intersection of Two Linked Lists (LC #160)

Find the node where two linked lists intersect (share the same physical node, not just same value).

```
List A: 4 → 1 ↘
                 8 → 4 → 5
List B: 5 → 6 → 1 ↗

Output: Node with value 8
```

**Elegant two-pointer approach:** Pointer A walks through list A then switches to list B. Pointer B walks through list B then switches to list A. They'll meet at the intersection!

Why? Because both pointers travel the same total distance: `len(A) + len(B)`. The "difference in length" is absorbed by the switch.

In [ ]:
def getIntersectionNode(headA, headB):
    """
    Two pointers that switch lists when they reach the end.
    Time: O(n + m), Space: O(1)
    """
    a, b = headA, headB
    
    # If they intersect, they'll meet. If not, both become None.
    while a != b:
        a = a.next if a else headB  # Switch to list B when A ends
        b = b.next if b else headA  # Switch to list A when B ends
    
    return a  # Either the intersection node or None

# Build intersecting lists: shared tail
shared = build_list([8, 4, 5])

# List A: 4 → 1 → [shared]
a1 = ListNode(4)
a2 = ListNode(1)
a1.next = a2
a2.next = shared

# List B: 5 → 6 → 1 → [shared]
b1 = ListNode(5)
b2 = ListNode(6)
b3 = ListNode(1)
b1.next = b2; b2.next = b3; b3.next = shared

result = getIntersectionNode(a1, b1)
print(f"Intersection at node with value: {result.val}")  # 8

This is one of the most elegant algorithms in all of DSA. The symmetry is perfect — both pointers travel the same total distance, and the length difference gets "equalized" by the switch.

---

## 🗺️ Merge & Sort Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  MERGE & SORT PATTERNS                                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. SORT LIST: Merge sort — find mid, split, sort halves, merge    ║
║     O(n log n), natural fit for linked lists (no extra space)       ║
║                                                                      ║
║  2. MERGE K LISTS:                                                   ║
║     • Heap: push heads, pop smallest, push next → O(N log k)       ║
║     • D&C: pair up lists, merge pairs like a bracket → O(N log k)  ║
║                                                                      ║
║  3. INTERSECTION: Two pointers that switch lists at the end         ║
║     Both travel len(A) + len(B) — meet at intersection or None     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 148 | Sort List | Medium | Merge sort |
| 21 | Merge Two Sorted Lists | Easy | Merge |
| 23 | Merge K Sorted Lists | Hard | Heap / D&C merge |
| 160 | Intersection of Two Lists | Easy | Pointer switch trick |
| 86 | Partition List | Medium | Two dummy heads |
| 328 | Odd Even Linked List | Medium | Separate & concat |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Merge sort is THE sorting algorithm for linked lists            │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Merge K lists: heap O(N log k) or divide & conquer O(N log k)  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Intersection trick: switch lists at end → both travel same dist│
├──────────────────────────────────────────────────────────────────────┤
│  📌 When in doubt about splitting: fast = head.next avoids edge bug│
├──────────────────────────────────────────────────────────────────────┤
│  📌 Heap needs (val, tiebreaker, node) — Python can't compare nodes│
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Linked List Problems

LRU Cache, Copy Random Pointer, and Add Two Numbers.

---

### 🎉 Merge & Sort Patterns Complete!
Sorting, merging K lists, finding intersections — the power trio.